# 推理引擎源码与前沿优化 · 第 5/8 课：稀疏推理：权重、MoE、Sparse Attention 与 HiSparse

> 状态：**未开始**  
> 源码审阅日期：2026-08-12；vLLM `8e958902eee5`；SGLang `9deb6952afa4`。

## 本课目标与通过标准

本课产出：能区分三类稀疏性的数学对象、kernel/通信前提和质量风险，并从 SGLang HiSparse 源码解释稀疏 attention 为什么仍需完整逻辑索引与多级 KV 状态。

通过要求：唯一代码填空题通过全部断言；Q1～Q3 都能沿源码对象给出因果链；能指出一个正确性不变量、一个性能边界和一个需要 benchmark 才能确认的结论；总分至少 8/10。

## 源码版本与阅读方法

本课程使用固定 commit 的永久链接保证行号可复现，同时链接 current docs 供核对最新变化。阅读时先画调用图和状态所有权，再进入分支；不要从大文件第一行机械顺读。

源码与论文阅读路线：

1. SGLang `scheduler.py` 搜索 `enable_hisparse`：看 prefill staging 如何转入 sparse decode。
2. `mem_cache/allocator/hisparse.py`：区分 full logical indices、compressed indices 与 sparse device buffer。
3. `hisparse_memory_pool.py`：追 index translation 与 KV transfer。
4. NSA 论文：只读三分支结构、hardware-aligned block selection 与训练方式。
5. SparseGPT/Wanda：比较权重评分和重构；再核对目标 runtime 是否真的有 sparse GEMM。
6. MoE/DeepEP：把“每 token 激活少量专家”与 all-to-all dispatch/combine 一起分析。

源码快照：SGLang `9deb6952afa4`。

## 核心对象

稀疏不是一个统一开关：

- 权重稀疏：参数矩阵中很多元素为零；unstructured 易获得质量，2:4 等半结构化模式更可能匹配硬件。
- 激活/专家稀疏：MoE 每 token 只路由 top-k experts，但所有专家权重仍需分布式存储，代价转为 dispatch、负载均衡和小 GEMM。
- Attention 稀疏：每个 query 只访问部分历史 KV。NSA 结合压缩全局分支、细粒度选择分支和滑动窗口分支，并从训练阶段适配该结构。

三者分别稀疏 weight edges、expert execution 和 token-token edges，不能用同一个 sparsity ratio 比性能。

## 调用链与状态变化

HiSparse 路径在 prefill 阶段保留构造 sparse decode 所需的状态，coordinator 收集 ready requests 后由 scheduler 构造 decode batch。allocator 同时维护完整逻辑位置、压缩位置和 sparse-device 位置之间的映射；decode 只把选中/窗口所需 KV 放在快速设备路径，但请求长度、因果位置和 cache 生命周期仍按完整序列维护。

MoE 的 top-k gate 产生 token→expert 路由，EP 先 dispatch 到专家所在 rank，执行 grouped GEMM，再 combine 回原 token 顺序。若 expert 负载偏斜或通信占主导，参数 FLOP 稀疏并不等于端到端加速。

## 正确性条件与常见误区

权重置零只有在存储格式、kernel 和硬件共同跳过零时才省时；否则 dense GEMM 仍做同样工作。稀疏 attention 只能访问因果历史，selection index 必须去重、边界合法，并与训练时模式兼容。MoE capacity、drop/overflow 和专家顺序必须保持模型语义。

质量评估不能只看 perplexity：还要覆盖长上下文检索、代码、推理、稀有专家和分布漂移。论文中的 kernel speedup不能直接外推到服务 goodput。

## 当前前沿与工程取舍

当前先进方向更偏向算法—系统协同：NSA/Flash Sparse Attention 等使用 block/hardware-aligned pattern；SGLang HiSparse 将稀疏访问与分层 KV/transfer 结合；MoE 使用高吞吐/低延迟 EP 通信和动态负载均衡；传统 unstructured pruning 仍有压缩价值，但主流通用 serving runtime 未必提供可移植加速。不存在跨模型/硬件统一“最先进”：必须分别给出质量、kernel latency、KV 容量、batch goodput 和通信证据。

## 具体推演

query position=15、block size=4、选择 blocks `[1,3]`、local window=3、sink=2：raw selected 分支访问 block 1 的 4～7、block 3 中不超过因果位置的 12～15，再与 local 13～15 和 sink 0～1 合并去重。NSA 的压缩分支还会读取 summary KV；本练习只模拟细粒度/窗口/sink 的 raw KV 访问集合。

请先口头复述“输入 → 状态所有者 → 状态迁移 → 输出/指标”，再做练习。

## 实践任务：唯一代码填空题

实现一个硬件友好的 block-sparse raw-KV 访问计划，合并 sink、selected blocks 和 local window，并严格保持 causal。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def sparse_raw_kv_indices(query_pos, selected_blocks, block_size,
                          local_window, sink_tokens=0):
    if query_pos < 0 or block_size <= 0 or local_window < 0 or sink_tokens < 0:
        raise ValueError("invalid sparse attention geometry")
    indices = set(range(min(sink_tokens, query_pos + 1)))

    # TODO：展开选中 block，但不得访问 query_pos 之后的 token。
    for block in selected_blocks:
        if block < 0:
            raise ValueError("negative block")
        start = block * block_size
        end = ______
        indices.update(______)

    local_start = max(0, query_pos - local_window + 1)
    indices.update(range(local_start, query_pos + 1))
    return sorted(indices)

assert sparse_raw_kv_indices(15, [1, 3], 4, 3, 2) == [0, 1, 4, 5, 6, 7, 12, 13, 14, 15]
assert sparse_raw_kv_indices(2, [1], 4, 2, 1) == [0, 1, 2]


### 检查方法

运行断言；加入重复 block 和完全位于未来的 block，确认去重且不越过 causal 边界。

### Q1

为什么 50% unstructured weight sparsity 不等于 GEMM 快 2 倍？

**你的答案：**


### Q2

NSA/HiSparse 为什么不能简单删除未选中的 KV cache？

**你的答案：**


### Q3

MoE 每 token 只激活 top-2 experts，为什么高并发 decode 仍可能比 dense 慢？

**你的答案：**


## 评分规则

- 代码 4 分：主路径 2 分、边界条件 1 分、能映射回源码对象 1 分；
- Q1～Q3 各 2 分：必须包含对象、状态变化、正确性或成本链；
- 一票否决：把论文峰值写成普遍生产结论；把源码支持写成所有模型/硬件可用；混淆算法正确性与性能；只背类名而说不清状态所有权。

## 参考资料

- [SGLang HiSparse scheduler path](https://github.com/sgl-project/sglang/blob/9deb6952afa483e38f96385a375b96f463da5303/python/sglang/srt/managers/scheduler.py#L2962-L3060)
- [SGLang HiSparse allocator](https://github.com/sgl-project/sglang/blob/9deb6952afa483e38f96385a375b96f463da5303/python/sglang/srt/mem_cache/allocator/hisparse.py)
- [Native Sparse Attention paper](https://arxiv.org/abs/2502.11089)
- [NVIDIA cuDNN NSA API](https://docs.nvidia.com/deeplearning/cudnn/latest/fe-oss-apis/nsa.html)
- [SparseGPT](https://arxiv.org/abs/2301.00774)
- [Wanda](https://arxiv.org/abs/2306.11695)
- [DeepEP](https://github.com/deepseek-ai/DeepEP)

源码链接固定到本课审阅 commit；current docs、支持矩阵和默认参数会变化，面试或部署前必须按目标版本重新核对。